<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "686"
iso_country = "Senegal"
year = 2025
import_portfolio = "h6_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 37.010*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Senegal': 

Number of trade records present in dataset: 37183
Number of partner countries present in dataset: 163
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$7723.7m
Total weight within the dataset: 9503.3 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,686,Senegal,724,Spain,Import,Sea,110811,110811 | Starch; wheat,1108,1108 | Starches; inulin,11,11 | Products of the mill,2322,Starches; inulin; wheat gluten; dextrins and o...,Agriculture,25,OFD - Food products n.e.c.,5785.12615,NaN,5785.12615,kg,7100.00,7100.00,9568.020139,2777.0,1725.547267,16510.071002,['Container'],['Container'],53.754609,53.541573,53.557076,53.557076,53.694566,53.694566,56.534911,107.352414,62.659009,57.391173,57.623576,59.368384,55.221496,101.945528,65.124537,59.762791,58.383665,61.385844,54.069970,112.672710,81.543427,71.042184,65.622978,64.178211,52.934569,119.427390,95.477453,89.19022,70.180812,69.087069,51.820504,122.945519,115.119581,117.068075,75.511823,72.702660,50.685303,120.140982,116.560356,122.993527,92.450449,74.086692,49.589104,106.483952,108.519202,115.205423,86.468016,74.996572,48.504283,95.105247,97.823978,113.782871,92.817058,75.596264,2322
1,1,686,Senegal,012,Algeria,Import,Sea,110811,110811 | Starch; wheat,1108,1108 | Starches; inulin,11,11 | Products of the mill,2322,Starches; inulin; wheat gluten; dextrins and o...,Agriculture,25,OFD - Food products n.e.c.,19.05498,NaN,19.05498,kg,12.78,12.78,31.515031,3248.0,2018.213008,63.604046,['Container'],['Container'],0.207086,0.206266,0.206325,0.206325,0.206855,0.206855,0.217797,0.413569,0.241390,0.221096,0.221991,0.228713,0.212737,0.392739,0.250888,0.230233,0.224920,0.236485,0.208301,0.434065,0.314141,0.273686,0.252809,0.247243,0.203927,0.460087,0.367821,0.34360,0.270367,0.266154,0.199635,0.473640,0.443491,0.450998,0.290905,0.280083,0.195262,0.462836,0.449042,0.473825,0.356160,0.285414,0.191039,0.410223,0.418064,0.443822,0.333113,0.288920,0.186860,0.366387,0.376861,0.438342,0.357572,0.291230,2322


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $1481.6m (19.2%), ... 

                              cifvalue    vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                              
10 | Cereals               632666143.0  1.657029e+09      8.2     17.4
19 | Preparations of cere  179629534.1  5.592240e+07      2.3      0.6
15 | Animal, vegetable or  126665995.5  1.315176e+08      1.6      1.4
17 | Sugars and sugar con  117835443.3  2.407513e+08      1.5      2.5
04 | Dairy produce; birds   70961276.0  3.069729e+07      0.9      0.3
03 | Fish and crustaceans   57300718.9  2.690046e+07      0.7      0.3
21 | Miscellaneous edible   51821780.3  1.294937e+07      0.7      0.1
07 | Vegetables and certa   45248134.9  7.999105e+07      0.6      0.8
23 | Food industries, res   42889643.9  8.690027e+07      0.6      0.9
08 | Fruit and nuts, edib   29313175.1  2.831914e+07      0.4      0.3
20 | Preparations of vege   25993502.1  2.0301

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $1178.4m (15.3%), ... 

                     cifvalue    vol_kg_vtw  pct_val  pct_vol
eora_desc                                                    
Agriculture       716289775.1  1.140607e+09      9.3     12.0
Food & Beverages  457204148.4  4.736562e+08      5.9      5.0
Fishing             4931900.0  2.035666e+06      0.1      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $1157.3m (15.0%), ... 

                                  cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                
PCR - Processed rice           353281529.4  613650016.5      4.6      6.5
OFD - Food products n.e.c.     325869222.5  152045777.2      4.2      1.6
VOL - Vegetable oils and fats  152287756.0  197383406.8      2.0      2.1
SGR - Sugar                    105339473.1  230978532.4      1.4      2.4
GRO - Cereal grains n.e.c.      59507422.4  246116725.7      0.8      2.6
V_F - Vegetables, fruit, nuts   51912878.4   88341483.9      0.7      0.9
MIL - Dairy products            51883897.0   19518286.0      0.7      0.2
OCR - Crops n.e.c.              30183281.6    6661600.8      0.4      0.1
CMT - Bovine meat prods         19109916.6    5449098.1      0.2      0.1
OMT - Meat products n.e.c.       4795044.5    1131652.4      0.1      0.0
OSD - Oil seeds                  2613990.1   

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 19.2% 
GTAP 15.0% 
EORA 15.3%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46413/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,686,Senegal,724,Spain,Import,Sea,110811,110811 | Starch; wheat,1108,1108 | Starches; inulin,11,11 | Products of the mill,2322,Starches; inulin; wheat gluten; dextrins and o...,Agriculture,25,OFD - Food products n.e.c.,5785.12615,NaN,5785.12615,kg,7100.00,7100.00,9568.020139,2777.0,1725.547267,16510.071002,['Container'],['Container'],53.754609,53.541573,53.557076,53.557076,53.694566,53.694566,56.534911,107.352414,62.659009,57.391173,57.623576,59.368384,55.221496,101.945528,65.124537,59.762791,58.383665,61.385844,54.069970,112.672710,81.543427,71.042184,65.622978,64.178211,52.934569,119.427390,95.477453,89.19022,70.180812,69.087069,51.820504,122.945519,115.119581,117.068075,75.511823,72.702660,50.685303,120.140982,116.560356,122.993527,92.450449,74.086692,49.589104,106.483952,108.519202,115.205423,86.468016,74.996572,48.504283,95.105247,97.823978,113.782871,92.817058,75.596264,2322,Processed Cereals
1,1,686,Senegal,012,Algeria,Import,Sea,110811,110811 | Starch; wheat,1108,1108 | Starches; inulin,11,11 | Products of the mill,2322,Starches; inulin; wheat gluten; dextrins and o...,Agriculture,25,OFD - Food products n.e.c.,19.05498,NaN,19.05498,kg,12.78,12.78,31.515031,3248.0,2018.213008,63.604046,['Container'],['Container'],0.207086,0.206266,0.206325,0.206325,0.206855,0.206855,0.217797,0.413569,0.241390,0.221096,0.221991,0.228713,0.212737,0.392739,0.250888,0.230233,0.224920,0.236485,0.208301,0.434065,0.314141,0.273686,0.252809,0.247243,0.203927,0.460087,0.367821,0.34360,0.270367,0.266154,0.199635,0.473640,0.443491,0.450998,0.290905,0.280083,0.195262,0.462836,0.449042,0.473825,0.356160,0.285414,0.191039,0.410223,0.418064,0.443822,0.333113,0.288920,0.186860,0.366387,0.376861,0.438342,0.357572,0.291230,2322,Processed Cereals


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $1175.5m of $7723.7m (15.2%)
Volume: 2161.1 of 9503.3 million kilograms (22.7%)
Value-to-Weight: 0.54 from 0.81 US$ per kg (+-33.1%)
Average distance to market (volume-weighted): 5270.2 from 4752.6 nm (+10.9%)

            food_group  N_com     cifvalue    vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    836  580786048.2  6.971462e+08      7.5      7.3
1  Unprocessed Cereals     64  279369251.3  1.043352e+09      3.6     11.0
2      Sugar and Honey    137  117852858.2  2.407563e+08      1.5      2.5
3           Vegetables    551   64184031.5  8.781699e+07      0.8      0.9
4        Fats and Oils    160   51362145.1  4.283318e+07      0.7      0.5
5                Dairy    229   51299610.0  2.047022e+07      0.7      0.2
6               Fruits    272   29217947.8  2.827590e+07      0.4      0.3
7        Meat and Prep     64    1462881.3  4.061047e+05      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,2.213161e+07,4.196690e+07,2.451895e+07,2.246348e+07,2.255437e+07,2.323804e+07,2.029085e+07,4.806894e+07,4.501361e+07,4.578657e+07,2.954860e+07,2.844485e+07,1.899611e+07,3.720106e+07,3.826139e+07,4.449775e+07,3.630262e+07,2.957552e+07
1,Fats and Oils,8.143097e+07,1.451957e+08,8.870079e+07,8.207717e+07,8.237841e+07,8.508965e+07,7.540701e+07,1.653677e+08,1.568825e+08,1.612492e+08,1.066548e+08,1.022331e+08,7.117431e+07,1.313939e+08,1.346928e+08,1.560752e+08,1.280380e+08,1.058944e+08
2,Fruits,3.134241e+07,5.674283e+07,3.425231e+07,3.166046e+07,3.178540e+07,3.277630e+07,2.895903e+07,6.535586e+07,6.134350e+07,6.290004e+07,4.131290e+07,3.952241e+07,2.728907e+07,5.139147e+07,5.269504e+07,6.099923e+07,4.994147e+07,4.104547e+07
3,Meat and Prep,6.569606e+05,1.247482e+06,7.281253e+05,6.669107e+05,6.696114e+05,6.898868e+05,6.021771e+05,1.428681e+06,1.337740e+06,1.360383e+06,8.774807e+05,8.448370e+05,5.636412e+05,1.105165e+06,1.136758e+06,1.322207e+06,1.078575e+06,8.784619e+05
4,Processed Cereals,1.837476e+09,2.891367e+09,1.935921e+09,1.832291e+09,1.837680e+09,1.899601e+09,1.735699e+09,3.474680e+09,3.302279e+09,3.416011e+09,2.410811e+09,2.184297e+09,1.664546e+09,2.829865e+09,2.868369e+09,3.260092e+09,2.761904e+09,2.240754e+09
5,Sugar and Honey,2.922483e+08,5.219724e+08,3.183571e+08,2.948538e+08,2.959360e+08,3.052429e+08,2.707153e+08,6.067968e+08,5.708575e+08,5.837155e+08,3.879436e+08,3.659732e+08,2.555927e+08,4.772221e+08,4.885275e+08,5.640224e+08,4.657440e+08,3.788595e+08
6,Unprocessed Cereals,1.470310e+09,2.737787e+09,1.620470e+09,1.489589e+09,1.495437e+09,1.541287e+09,1.352362e+09,3.150263e+09,2.954158e+09,3.009388e+09,1.959924e+09,1.874604e+09,1.269435e+09,2.449761e+09,2.515950e+09,2.919493e+09,2.390839e+09,1.946450e+09
7,Vegetables,1.165222e+08,2.198734e+08,1.289003e+08,1.182110e+08,1.186875e+08,1.222944e+08,1.069223e+08,2.520266e+08,2.360658e+08,2.402991e+08,1.553719e+08,1.494609e+08,1.001673e+08,1.953456e+08,2.008553e+08,2.334903e+08,1.905644e+08,1.553780e+08


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.39,0.05,0.01,0.01,0.02,0.54,0.48,0.50,0.18,0.16,0.35,0.38,0.50,0.34,0.21
1,Fats and Oils,1.24,0.14,0.01,0.02,0.07,1.75,1.59,1.67,0.61,0.52,1.17,1.24,1.65,1.11,0.68
2,Fruits,0.87,0.10,0.01,0.02,0.05,1.25,1.11,1.16,0.42,0.36,0.82,0.87,1.15,0.78,0.47
3,Meat and Prep,0.40,0.05,0.01,0.01,0.02,0.56,0.50,0.52,0.19,0.17,0.37,0.39,0.52,0.35,0.22
4,Processed Cereals,1.81,0.17,-0.01,0.00,0.11,2.99,2.70,2.89,1.16,0.77,2.01,2.07,2.75,1.89,0.99
5,Sugar and Honey,1.95,0.22,0.02,0.03,0.11,2.85,2.55,2.66,0.99,0.81,1.88,1.98,2.62,1.78,1.05
6,Unprocessed Cereals,4.54,0.54,0.07,0.09,0.25,6.44,5.73,5.93,2.17,1.87,4.22,4.46,5.91,4.01,2.42
7,Vegetables,1.61,0.19,0.03,0.03,0.09,2.26,2.01,2.08,0.75,0.66,1.48,1.57,2.08,1.41,0.86
